In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. 比較したい複数の投資先（ティッカーシンボル）を指定
# VOO: S&P500(株式), TLT: 米国債(債券), GLD: 金(コモディティ), BTC-USD: ビットコイン(暗号資産)
tickers = ["VOO", "TLT", "GLD", "BTC-USD"]

print("=== 複数銘柄のデータ取得を開始 ===")

# 2. 過去5年分の終値（Close）を一括取得
data = yf.download(tickers, period="5y")['Close']

# 3. 各銘柄の日次リターン（前日比の変化率）を計算
daily_returns = data.pct_change().dropna()

# 4. 各投資ごとのリターンとリスクを計算（年率換算）
summary_dict = {}

for ticker in tickers:
    # 期待リターン（日次平均 × 年間営業日252）
    # ※暗号資産(BTC)は365日動いていますが、比較のため一律252日で換算（またはBTCのみ365日でも可）
    ann_return = daily_returns[ticker].mean() * 252

    # リスク＝標準偏差（日次標準偏差 × √252）
    ann_risk = daily_returns[ticker].std() * np.sqrt(252)

    # シャープ・レシオ（効率性：リスク1単位あたりに得られるリターン）
    sharpe = ann_return / ann_risk if ann_risk != 0 else 0

    summary_dict[ticker] = {
        "年率リターン (%)": round(ann_return * 100, 2),
        "年率リスク (%)": round(ann_risk * 100, 2),
        "シャープ・レシオ": round(sharpe, 2)
    }

# 5. 結果をデータフレームにして見やすく表示
summary_df = pd.DataFrame(summary_dict).T
print("\n--- 投資ごとのリターンとリスク一覧 ---")
print(summary_df)

=== 複数銘柄のデータ取得を開始 ===


[*********************100%***********************]  4 of 4 completed


--- 投資ごとのリターンとリスク一覧 ---
         年率リターン (%)  年率リスク (%)  シャープ・レシオ
VOO           13.09      17.41      0.75
TLT            7.16      15.67      0.46
GLD           17.16      17.96      0.96
BTC-USD        8.73      47.98      0.18


_IncompleteInputError: incomplete input (152855823.py, line 67)

In [5]:
import numpy as np
import pandas as pd
import yfinance as yf

# 投資信託の代わりに、同じ値動きをする主要ETFを指定（13アセットに拡充）
ticker_dict = {
    # --- 株式（インデックス） ---
    "VT": "全世界株式（オルカン等）",
    "VTI": "全米株式（S&P500/CRSP）",
    "QQQ": "米国ナスダック100（NASDAQ）",
    "EEM": "新興国株式（全体）",
    "EPI": "インド株式",
    "1306.T": "日本株式（TOPIX）",

    # --- 株式（高配当・スマートベータ） ---
    "VYM": "米国高配当株式",

    # --- 債券 ---
    "BNDX": "先進国債券（除く米・為替ヘッジ有）",
    "AGG": "米国総合債券",

    # --- 不動産（REIT） ---
    "1343.T": "J-REIT（日本不動産信託）",
    "VNQ": "米国REIT（米国不動産）",
    "RWX": "先進国REIT（除く米国）",

    # --- コモディティ（実物資産） ---
    "GLD": "金（ゴールド）"
}

tickers = list(ticker_dict.keys())

print("=== 拡大アセットのデータ取得を開始 ===")

# 過去5年分の終値（Close）を一括取得
data = yf.download(tickers, period="5y")['Close']

# 祝日のズレ（欠損値）を前日データで埋めてから変化率を計算
daily_returns = data.ffill().pct_change().dropna()

summary_dict = {}

for ticker in tickers:
    # 年率換算リターン（営業日252日）
    ann_return = daily_returns[ticker].mean() * 252

    # 年率換算リスク（標準偏差 × √252）
    ann_risk = daily_returns[ticker].std() * np.sqrt(252)

    # シャープ・レシオ
    sharpe = ann_return / ann_risk if ann_risk != 0 else 0

    asset_name = ticker_dict[ticker]
    summary_dict[asset_name] = {
        "ティッカー": ticker,
        "年率リターン (%)": round(ann_return * 100, 2),
        "年率リスク (%)": round(ann_risk * 100, 2),
        "シャープ・レシオ": round(sharpe, 2)
    }

# データフレームにして表示
summary_df = pd.DataFrame(summary_dict).T
summary_df = summary_df[["ティッカー", "年率リターン (%)", "年率リスク (%)", "シャープ・レシオ"]]

# リターンが高い順に並び替えて表示
summary_df = summary_df.sort_values(by="年率リターン (%)", ascending=False)

print("\n--- 全13アセットのリスク・リターン一覧（リターン順） ---")
print(summary_df)

=== 拡大アセットのデータ取得を開始 ===


[*********************100%***********************]  13 of 13 completed


--- 全13アセットのリスク・リターン一覧（リターン順） ---
                     ティッカー 年率リターン (%) 年率リスク (%) シャープ・レシオ
米国ナスダック100（NASDAQ）     QQQ      17.26     22.17     0.78
金（ゴールド）                GLD      17.01     17.92     0.95
全米株式（S&P500/CRSP）      VTI       12.5     17.18     0.73
米国高配当株式                VYM      11.51     13.75     0.84
全世界株式（オルカン等）            VT      10.96     15.88     0.69
新興国株式（全体）              EEM       7.79     18.94     0.41
インド株式                  EPI       6.35     15.96      0.4
米国REIT（米国不動産）          VNQ        3.9     18.52     0.21
J-REIT（日本不動産信託）     1343.T       1.27     12.24      0.1
先進国債券（除く米・為替ヘッジ有）     BNDX       0.44       4.8     0.09
米国総合債券                 AGG       0.31      5.99     0.05
先進国REIT（除く米国）          RWX      -1.45     15.59    -0.09
日本株式（TOPIX）         1306.T      -1.64     43.87    -0.04


In [8]:
import numpy as np
import pandas as pd
import yfinance as yf

ticker_dict = {
    "VT":     "全世界株式",
    "VTI":    "全米株式",
    "QQQ":    "ナスダック100",
    "EEM":    "新興国株式",
    "EPI":    "インド株式",
    "1306.T": "日本株式（TOPIX）",
    "VYM":    "米国高配当株式",
    "BNDX":   "先進国債券",
    "AGG":    "米国総合債券",
    "1343.T": "J-REIT",
    "VNQ":    "米国REIT",
    "RWX":    "先進国REIT",
    "GLD":    "金",
}
tickers = list(ticker_dict.keys())

print("=== 20年分データ取得 ===")
raw = yf.download(tickers, start="2005-01-01", end="2025-12-31",
                  auto_adjust=True)["Close"]
raw = raw.ffill().bfill()
available = [t for t in tickers if t in raw.columns and raw[t].notna().sum() > 252]
raw = raw[available]

prices        = raw.to_numpy(dtype=np.float64)
daily_returns = np.diff(prices, axis=0) / prices[:-1]
dates         = raw.index.to_numpy()
available_names = [ticker_dict[t] for t in available]

# =============================================
# 年別×銘柄別 集計
# =============================================
TRADING_DAYS = 252

# 日次リターンに日付を付けて DataFrame 化（dates は prices と同じ長さなので
# daily_returns の日付は prices の2行目以降に対応）
ret_df = pd.DataFrame(
    daily_returns,
    index  = pd.DatetimeIndex(dates[1:]),   # 1行目は diff で消えるため +1
    columns= available,
)

# --- 年でグループ化 ---
grouped = ret_df.groupby(ret_df.index.year)

# 年率リターン（日次平均 × 252）
annual_return = grouped.mean() * TRADING_DAYS * 100          # 単位: %

# 年率リスク（日次標準偏差 × √252）
annual_risk   = grouped.std()  * np.sqrt(TRADING_DAYS) * 100 # 単位: %

# シャープ・レシオ（無リスク金利は簡略化のため 0 で計算）
annual_sharpe = grouped.mean() / grouped.std()

# 列名を日本語に変換
annual_return.columns  = available_names
annual_risk.columns    = available_names
annual_sharpe.columns  = available_names

annual_return  = annual_return.round(2)
annual_risk    = annual_risk.round(2)
annual_sharpe  = annual_sharpe.round(3)

# =============================================
# 出力
# =============================================
pd.set_option("display.max_columns", None)
pd.set_option("display.width",       200)

print("\n" + "=" * 80)
print("【年率リターン(%)  ―  行: 年, 列: アセット】")
print("=" * 80)
print(annual_return.to_string())

print("\n" + "=" * 80)
print("【年率リスク(%)    ―  行: 年, 列: アセット】")
print("=" * 80)
print(annual_risk.to_string())

print("\n" + "=" * 80)
print("【シャープ・レシオ ―  行: 年, 列: アセット】")
print("=" * 80)
print(annual_sharpe.to_string())

# =============================================
# 特定アセットを年ごとに縦に並べた形式（見やすい別形式）
# =============================================
print("\n" + "=" * 80)
print("【年別×アセット別 縦長テーブル（リターン・リスク・シャープ一覧）】")
print("=" * 80)

long = (
    annual_return
    .stack()
    .rename("リターン(%)")
    .to_frame()
    .join(annual_risk  .stack().rename("リスク(%)"))
    .join(annual_sharpe.stack().rename("シャープ"))
)
long.index.names = ["年", "アセット"]
print(long.to_string())

# =============================================
# 任意の年・アセットを取り出す例
# =============================================
print("\n--- 例: 2020年のリターン（高い順） ---")
print(annual_return.loc[2020].sort_values(ascending=False).to_string())

print("\n--- 例: QQQ（ナスダック100）の年別推移 ---")
qqq_name = ticker_dict["QQQ"]
print(
    pd.DataFrame({
        "リターン(%)": annual_return[qqq_name],
        "リスク(%)":   annual_risk[qqq_name],
        "シャープ":    annual_sharpe[qqq_name],
    }).to_string()
)

=== 20年分データ取得 ===


[*********************100%***********************]  13 of 13 completed



【年率リターン(%)  ―  行: 年, 列: アセット】
      全世界株式   全米株式  ナスダック100  新興国株式  インド株式  日本株式（TOPIX）  米国高配当株式  先進国債券  米国総合債券  J-REIT  米国REIT  先進国REIT      金
2005   0.00   7.41      3.49  31.26   0.00         0.00     0.00   0.00    2.30    0.00   13.72     0.00  19.01
2006   0.00  15.19      8.15  31.04   0.00         0.00     2.83   0.00    3.89    0.00   31.40     4.27  23.32
2007   0.00   6.50     19.14  34.26   0.00         0.00     2.41   0.00    6.49    0.00  -14.35    -5.21  28.27
2008 -30.86 -36.79    -44.42 -41.14 -63.55       -39.02   -30.10   0.00    8.05  -15.95  -18.96   -59.63   9.79
2009  32.05  28.14     45.46  58.37  75.59         7.54    19.46   0.00    3.00    3.40   45.42    35.71  22.96
2010  14.07  17.22     19.59  18.00  20.91         0.65    14.15   0.00    6.05   24.48   27.74    21.75  26.23
2011  -4.13   3.72      6.03 -15.09 -45.83       -17.96    11.50   0.00    7.26  -26.95   12.06   -12.21  10.89
2012  16.41  15.60     17.26  18.71  24.75        17.56    12.14   0.00  

In [9]:
# 特定の年を取り出す
annual_return.loc[2008]          # 2008年（リーマンショック）

# 特定の銘柄を取り出す
annual_return["ナスダック100"]   # QQQの年別リターン

# 両方指定
annual_return.loc[2020, "金"]    # 2020年のGLD

np.float64(23.25)

In [ ]:
##ここで学習を行う
